<a href="https://colab.research.google.com/github/jeremydouti2-hub/Data-Science-Projects/blob/main/credit-score/Credit_Score_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import files
uploded = files.upload()

In [ ]:
import pandas as pd

# load dataset
df = pd.read_csv('tcs project.csv')

# view 5 rows dataset
df.head()

In [ ]:
# ============================================================
# SECTION 1 — DATA PREPARATION
# Project: Predictive Modeling of Credit Scores Through Logistic Regression
# Dataset : tcs project.csv
# ============================================================

# ── 1.1  Install PySpark (Google Colab only)
# !pip install pyspark

# ── 1.2  Import Libraries
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.stat import Correlation
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# ── 1.3  Create Spark Session
spark = SparkSession.builder \
    .appName("CreditScorePrediction") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(" Spark Session created successfully")
print(f"   Spark version : {spark.version}")

# ── 1.4  Load Dataset
# Upload 'tcs project.csv' in Google Colab before running this cell
df = spark.read.csv(
    "tcs project.csv",
    header=True,
    inferSchema=True
)

print(f"\n Dataset loaded — {df.count()} rows × {len(df.columns)} columns")

# ── 1.5  Initial Exploration
print("\n Schema:")
df.printSchema()

print("\n First 5 rows:")
df.show(5, truncate=False)

print("\n Summary Statistics:")
df.describe().show()

# ── 1.6  Drop Unnamed Index Column
# The CSV has an auto-index column '' or '_c0' — drop it
index_col = [c for c in df.columns if c in ('', '_c0')]
if index_col:
    df = df.drop(*index_col)
    print(f"\n Dropped index column(s): {index_col}")

# ── 1.7  Check & Handle Missing Values
print("\n Missing values per column:")
missing = df.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]
)
missing.show()

# Drop rows with any null (dataset is clean but best practice)
df = df.dropna()
print(f" After dropna : {df.count()} rows remaining")

# ── 1.8  Cast All Columns to DoubleType
feature_cols = ['age', 'education', 'n_yrs_employed', 'n_yrs_in_address',
                'income', 'debt_to_income_ratio', 'credit_to_debit_ratio',
                'other_debit']
label_col    = 'defaulted_loan'   # Binary target  (0 = no default, 1 = default)

for c in feature_cols + [label_col]:
    df = df.withColumn(c, F.col(c).cast(DoubleType()))

print("\n All columns cast to DoubleType")

# ── 1.9  Exploratory Visualisation (Pandas bridge)
pdf = df.toPandas()

# Distribution of target
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

pdf['defaulted_loan'].value_counts().plot(
    kind='bar', ax=axes[0], color=['steelblue', 'tomato'],
    title='Class Distribution (0=No Default, 1=Default)'
)
axes[0].set_xlabel("Defaulted Loan")
axes[0].set_ylabel("Count")

# Credit score distribution
axes[1].hist(pdf['credit_score'], bins=30, color='steelblue', edgecolor='black')
axes[1].set_title('Credit Score Distribution')
axes[1].set_xlabel("Credit Score")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("plot_distributions.png", dpi=150)
plt.show()
print(" Distribution plots saved")

# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 7))
corr_matrix = pdf[feature_cols + ['defaulted_loan', 'credit_score']].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm',
            linewidths=0.5, ax=ax)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("plot_correlation_heatmap.png", dpi=150)
plt.show()
print(" Correlation heatmap saved")

# Boxplots — features vs defaulted_loan
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(feature_cols):
    pdf.boxplot(column=col, by='defaulted_loan', ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel("Defaulted Loan")
plt.suptitle("Feature Distribution by Default Status")
plt.tight_layout()
plt.savefig("plot_boxplots.png", dpi=150)
plt.show()
print(" Boxplots saved")

# ── 1.10  Feature Engineering — PCA optional check
# VectorAssembler : combine all features into single 'features' vector
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)
df_assembled = assembler.transform(df)

# Standard Scaler — important for logistic regression convergence
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)
scaler_model = df_assembled
df_assembled  = assembler.transform(df)
scaler_fit    = scaler.fit(df_assembled)
df_scaled     = scaler_fit.transform(df_assembled)

# Keep only scaled features + label
df_final = df_scaled.select("scaled_features", F.col(label_col).alias("label"))

print("\n Features assembled & scaled")
print(f"   Final dataset : {df_final.count()} rows")
df_final.show(5, truncate=False)

# ── 1.11  Train / Test Split (75% / 25%)
train_data, test_data = df_final.randomSplit([0.75, 0.25], seed=42)

print(f"\n  Data split complete:")
print(f"   Training set : {train_data.count()} rows ({train_data.count()/df_final.count()*100:.1f}%)")
print(f"   Test set     : {test_data.count()} rows ({test_data.count()/df_final.count()*100:.1f}%)")

# ── 1.12  Save split datasets for reuse
train_data.write.mode("overwrite").parquet("train_data.parquet")
test_data.write.mode("overwrite").parquet("test_data.parquet")
print("\n  Train & Test datasets saved as Parquet files")

print("\n" + "="*55)
print("  SECTION 1 — DATA PREPARATION COMPLETE ")
print("="*55)

In [ ]:
# ============================================================
# SECTION 2 — BUILD MODEL
# Project: Predictive Modeling of Credit Scores Through Logistic Regression
# ============================================================

# ── 2.1  Import Libraries
from pyspark.sql import SparkSession
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np

# ── 2.2  Restart Spark Session
spark = SparkSession.builder \
    .appName("CreditScorePrediction_ModelBuilding") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(" Spark Session ready")

# ── 2.3  Load Train & Test Data (from Section 1)
train_data = spark.read.parquet("train_data.parquet")
test_data  = spark.read.parquet("test_data.parquet")

print(f" Train rows : {train_data.count()}")
print(f" Test  rows : {test_data.count()}")
train_data.show(5, truncate=False)

# ── 2.4  Build Logistic Regression Model
# featuresCol = 'scaled_features' (from VectorAssembler + StandardScaler in Section 1)
# labelCol    = 'label'           (defaulted_loan : 0 or 1)

logistic_regression = LogisticRegression(
    featuresCol="scaled_features",
    labelCol="label",
    maxIter=100,            # max iterations for solver convergence
    regParam=0.01,          # L2 regularization (Ridge) — reduces overfitting
    elasticNetParam=0.0,    # 0.0 = pure L2, 1.0 = pure L1
    threshold=0.5,          # decision boundary
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction"
)

# ── 2.5  Fit (Train) the Model
print("\n Training Logistic Regression model...")
model = logistic_regression.fit(train_data)
print(" Model trained successfully")

# ── 2.6  Examine Training Summary
training_summary = model.summary

print(f"\n Training Summary:")
print(f"   Total Iterations run     : {training_summary.totalIterations}")
print(f"   Training Accuracy        : {training_summary.accuracy:.4f}")
print(f"   Training AUC-ROC         : {training_summary.areaUnderROC:.4f}")

# Objective history (loss per iteration)
obj_history = training_summary.objectiveHistory
print(f"\n Objective (Loss) History — first 5 iterations:")
for i, loss in enumerate(obj_history[:5]):
    print(f"   Iteration {i+1} : {loss:.6f}")

# ── 2.7  Plot Objective History
plt.figure(figsize=(9, 4))
plt.plot(range(1, len(obj_history)+1), obj_history,
         marker='o', markersize=3, color='steelblue', linewidth=1.5)
plt.title("Training Loss (Objective History) — Logistic Regression")
plt.xlabel("Iteration")
plt.ylabel("Objective / Loss")
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("plot_objective_history.png", dpi=150)
plt.show()
print(" Objective history plot saved")

# ── 2.8  Examine Coefficients & Intercept
coefficients = model.coefficients.toArray()
intercept    = model.intercept

feature_cols = ['age', 'education', 'n_yrs_employed', 'n_yrs_in_address',
                'income', 'debt_to_income_ratio', 'credit_to_debit_ratio',
                'other_debit']

coef_df = pd.DataFrame({
    'Feature'    : feature_cols,
    'Coefficient': coefficients
}).sort_values('Coefficient', key=abs, ascending=False)

print(f"\n Model Intercept (Bias) : {intercept:.6f}")
print("\n Feature Coefficients (sorted by absolute importance):")
print(coef_df.to_string(index=False))

# ── 2.9  Feature Importance Bar Chart
colors = ['tomato' if c > 0 else 'steelblue' for c in coef_df['Coefficient']]

plt.figure(figsize=(10, 5))
bars = plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.axvline(x=0, color='black', linewidth=0.8)
plt.title("Feature Coefficients — Logistic Regression\n(Red = increases default risk | Blue = decreases default risk)")
plt.xlabel("Coefficient Value")
plt.tight_layout()
plt.savefig("plot_feature_coefficients.png", dpi=150)
plt.show()
print(" Feature coefficients plot saved")

# ── 2.10  Interpretation of Coefficients ─
print("\n Coefficient Interpretation:")
for _, row in coef_df.iterrows():
    direction = "↑ INCREASES" if row['Coefficient'] > 0 else "↓ DECREASES"
    print(f"   {row['Feature']:<28} {direction} default risk  (coef = {row['Coefficient']:+.4f})")

# ── 2.11  Odds Ratios (exp of coefficients)
coef_df['Odds_Ratio'] = np.exp(coef_df['Coefficient'])
print("\n Odds Ratios (exp of coefficients):")
print(coef_df[['Feature', 'Coefficient', 'Odds_Ratio']].to_string(index=False))

# ── 2.12  Make Predictions on Test Data
predictions = model.transform(test_data)

print("\n Sample Predictions:")
predictions.select("label", "prediction", "probability").show(10, truncate=False)

# ── 2.13  Training ROC Curve
roc_pdf = training_summary.roc.toPandas()

plt.figure(figsize=(7, 6))
plt.plot(roc_pdf['FPR'], roc_pdf['TPR'],
         color='darkorange', lw=2,
         label=f"ROC Curve (AUC = {training_summary.areaUnderROC:.4f})")
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Random Classifier')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Training ROC Curve — Logistic Regression")
plt.legend(loc="lower right")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("plot_training_roc_curve.png", dpi=150)
plt.show()
print(" Training ROC curve saved")

# ── 2.14  Precision-Recall Curve
pr_pdf = training_summary.pr.toPandas()

plt.figure(figsize=(7, 6))
plt.plot(pr_pdf['recall'], pr_pdf['precision'],
         color='green', lw=2, label="Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Training Precision-Recall Curve")
plt.legend(loc="upper right")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("plot_precision_recall_curve.png", dpi=150)
plt.show()
print(" Precision-Recall curve saved")

# ── 2.15  Save Model
model.write().overwrite().save("logit_model")
print("\n  Model saved to 'logit_model'")

# ── 2.16  Save predictions for Section 3
predictions.write.mode("overwrite").parquet("predictions.parquet")
print(" Predictions saved for Section 3")

print("\n" + "="*55)
print("  SECTION 2 — BUILD MODEL COMPLETE ")
print("="*55)
print(f"\n  Key Results:")
print(f"  • Training Accuracy  : {training_summary.accuracy:.4f}")
print(f"  • Training AUC-ROC   : {training_summary.areaUnderROC:.4f}")
print(f"  • Intercept (Bias)   : {intercept:.6f}")
print(f"  • Top Feature        : {coef_df.iloc[0]['Feature']} (coef={coef_df.iloc[0]['Coefficient']:+.4f})")

In [ ]:
# ============================================================
# SECTION 3 — EVALUATION & TUNE
# Project: Predictive Modeling of Credit Scores Through Logistic Regression
# ============================================================

# ── 3.1  Import Libraries
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.classification import LogisticRegression, LogisticRegressionModel
from pyspark.ml.evaluation import (BinaryClassificationEvaluator,
                                   MulticlassClassificationEvaluator)
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.ml.functions import vector_to_array

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, precision_recall_curve

# ── 3.2  Restart Spark Session ─
spark = SparkSession.builder \
    .appName("CreditScore_EvaluationTuning") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(" Spark Session ready")

# ── 3.3  Load Data & Predictions
train_data  = spark.read.parquet("train_data.parquet")
test_data   = spark.read.parquet("test_data.parquet")
predictions = spark.read.parquet("predictions.parquet")

model = LogisticRegressionModel.load("logit_model")

print(f" Train rows : {train_data.count()}")
print(f" Test rows  : {test_data.count()}")

# ============================================================
# PART A — BASELINE MODEL EVALUATION
# ============================================================

# ── 3.4  AUC
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

evaluator_pr = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

auc_roc = evaluator_auc.evaluate(predictions)
auc_pr  = evaluator_pr.evaluate(predictions)

print("\n BASELINE METRICS")
print("AUC ROC:", auc_roc)
print("AUC PR :", auc_pr)

# ── 3.5 Accuracy etc
eval_acc = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy")

eval_prec = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision")

eval_rec = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall")

eval_f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1")

accuracy  = eval_acc.evaluate(predictions)
precision = eval_prec.evaluate(predictions)
recall    = eval_rec.evaluate(predictions)
f1_score  = eval_f1.evaluate(predictions)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1_score)

# ── 3.6 Confusion Matrix (FIXED)
preds_labels = predictions.select(
    F.col("prediction").cast("double"),
    F.col("label").cast("double")
).rdd.map(lambda r: (r[0], r[1]))

cm = MulticlassMetrics(preds_labels).confusionMatrix().toArray()
cm = np.array(cm).astype(int)   # FIX

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix — Baseline")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# ── 3.7 ROC BASELINE
roc_df = predictions.select(
    vector_to_array("probability")[1].alias("score"),
    F.col("label").cast("double")
).toPandas()

fpr, tpr, _ = roc_curve(roc_df["label"], roc_df["score"])
precision_curve, recall_curve, _ = precision_recall_curve(
    roc_df["label"], roc_df["score"]
)

plt.figure(figsize=(7,6))
plt.plot(fpr, tpr, label=f"AUC={auc_roc:.4f}")
plt.plot([0,1],[0,1],'--')
plt.legend()
plt.title("ROC Curve — Baseline")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.show()

plt.figure(figsize=(7,6))
plt.plot(recall_curve, precision_curve)
plt.title("Precision Recall Curve — Baseline")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()

# ============================================================
# PART B — HYPERPARAMETER TUNING
# ============================================================

lr_tuning = LogisticRegression(
    featuresCol="scaled_features",
    labelCol="label",
    maxIter=200
)

param_grid = ParamGridBuilder() \
    .addGrid(lr_tuning.regParam, [0.001,0.01,0.05,0.1]) \
    .addGrid(lr_tuning.elasticNetParam,[0.0,0.2,0.5,0.8]) \
    .addGrid(lr_tuning.threshold,[0.4,0.5,0.6]) \
    .build()

cv = CrossValidator(
    estimator=lr_tuning,
    estimatorParamMaps=param_grid,
    evaluator=evaluator_auc,
    numFolds=3,
    seed=42
)

print("Running cross validation...")
cv_model = cv.fit(train_data)

best_model = cv_model.bestModel
best_predictions = best_model.transform(test_data)

# ── Tuned metrics
tuned_auc_roc  = evaluator_auc.evaluate(best_predictions)
tuned_accuracy = eval_acc.evaluate(best_predictions)
tuned_f1       = eval_f1.evaluate(best_predictions)

print("\nTUNED METRICS")
print("AUC:", tuned_auc_roc)
print("Accuracy:", tuned_accuracy)
print("F1:", tuned_f1)

# ── Confusion tuned (FIXED)
preds_labels_tuned = best_predictions.select(
    F.col("prediction").cast("double"),
    F.col("label").cast("double")
).rdd.map(lambda r: (r[0], r[1]))

cm_tuned = MulticlassMetrics(preds_labels_tuned).confusionMatrix().toArray()
cm_tuned = np.array(cm_tuned).astype(int)

plt.figure(figsize=(6,5))
sns.heatmap(cm_tuned, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix — Tuned")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# ── ROC tuned
roc_tuned_df = best_predictions.select(
    vector_to_array("probability")[1].alias("score"),
    F.col("label").cast("double")
).toPandas()

fpr_tuned, tpr_tuned, _ = roc_curve(
    roc_tuned_df["label"],
    roc_tuned_df["score"]
)

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, '--', label=f"Baseline {auc_roc:.3f}")
plt.plot(fpr_tuned, tpr_tuned, label=f"Tuned {tuned_auc_roc:.3f}")
plt.plot([0,1],[0,1],':')
plt.legend()
plt.title("ROC Curve — Baseline vs Tuned")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.show()

# ── Save model
best_model.write().overwrite().save("logit_model_tuned")
best_predictions.write.mode("overwrite").parquet("best_predictions.parquet")

print("\nSECTION 3 COMPLETE")
print("Best model saved → logit_model_tuned")

In [ ]:
# ============================================================
# SECTION 4 — DEPLOYMENT
# Project: Predictive Modeling of Credit Scores Through Logistic Regression
# ============================================================

# ── 4.1  Import Libraries
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.classification import LogisticRegressionModel
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.functions import vector_to_array
from pyspark.sql.types import DoubleType

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
import numpy as np

# ── 4.2  Spark Session
spark = SparkSession.builder \
    .appName("CreditScore_Deployment") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session ready")

# ── 4.3  Load Best Tuned Model ───────────────────────────────
loaded_model = LogisticRegressionModel.load("logit_model_tuned")

print("Tuned model loaded")
print("regParam        :", loaded_model.getRegParam())
print("elasticNetParam :", loaded_model.getElasticNetParam())
print("threshold       :", loaded_model.getThreshold())

# ── 4.4  Load Full Dataset
df_raw = spark.read.csv(
    "tcs project.csv",
    header=True,
    inferSchema=True
)

# Drop index column
index_col = [c for c in df_raw.columns if c in ('', '_c0')]
if index_col:
    df_raw = df_raw.drop(*index_col)

feature_cols = [
    'age','education','n_yrs_employed','n_yrs_in_address',
    'income','debt_to_income_ratio',
    'credit_to_debit_ratio','other_debit'
]

label_col = "defaulted_loan"

for c in feature_cols + [label_col]:
    df_raw = df_raw.withColumn(c, F.col(c).cast(DoubleType()))

# Assemble features
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df_assembled = assembler.transform(df_raw)

# Scale features
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)

scaler_fit = scaler.fit(df_assembled)
df_scaled  = scaler_fit.transform(df_assembled)

df_final = df_scaled.select(
    *feature_cols,
    F.col(label_col).alias("label"),
    "scaled_features",
    "credit_score"
)

# ── 4.5  Run Inference
inference_preds = loaded_model.transform(df_final)

# SAFE probability extraction
inference_preds = inference_preds.withColumn(
    "default_probability",
    vector_to_array("probability")[1]
)

print("Inference complete:", inference_preds.count())

pdf = inference_preds.select(
    *feature_cols,
    "label",
    "prediction",
    "default_probability",
    "credit_score"
).toPandas()

# ============================================================
# PART A — INFERENCE VISUALISATIONS
# ============================================================

# ── 4.6 Default Probability Distribution
fig, axes = plt.subplots(1,2,figsize=(14,5))

axes[0].hist(
    pdf[pdf['label']==0]['default_probability'],
    bins=30, alpha=0.7, label="No Default"
)

axes[0].hist(
    pdf[pdf['label']==1]['default_probability'],
    bins=30, alpha=0.7, label="Default"
)

axes[0].axvline(
    loaded_model.getThreshold(),
    linestyle="--",
    color="black"
)

axes[0].set_title("Default Probability Distribution")
axes[0].legend()

correct = (pdf['label']==pdf['prediction']).sum()
incorrect = len(pdf)-correct

axes[1].pie(
    [correct,incorrect],
    labels=["Correct","Incorrect"],
    autopct="%1.1f%%"
)

axes[1].set_title("Prediction Accuracy")

plt.tight_layout()
plt.show()

# ── 4.7 Credit Score vs Probability
plt.figure(figsize=(8,6))

plt.scatter(
    pdf['credit_score'],
    pdf['default_probability'],
    c=pdf['label'],
    cmap='RdYlGn_r',
    alpha=0.5
)

plt.axhline(
    loaded_model.getThreshold(),
    linestyle="--",
    color="black"
)

plt.xlabel("Credit Score")
plt.ylabel("Default Probability")
plt.title("Credit Score vs Default Probability")

plt.show()

# ── 4.8 Feature Impact
fig, axes = plt.subplots(2,4,figsize=(18,9))
axes = axes.flatten()

for i,col in enumerate(feature_cols):

    axes[i].scatter(
        pdf[col],
        pdf['default_probability'],
        alpha=0.4
    )

    z = np.polyfit(pdf[col], pdf['default_probability'],1)
    p = np.poly1d(z)

    x_line = np.linspace(
        pdf[col].min(),
        pdf[col].max(),
        100
    )

    axes[i].plot(x_line, p(x_line),'--')

    axes[i].set_title(col)

plt.tight_layout()
plt.show()

# ============================================================
# PART B — NEW DATA INFERENCE
# ============================================================

print("NEW SAMPLE PREDICTIONS")

new_data = [
    (35,3,10,5,80,12.5,2.5,3.1),
    (50,4,25,20,200,3.2,8.1,1.2),
    (23,1,2,1,20,35.0,0.5,7.8),
    (41,2,15,10,120,5.5,4.2,2.0),
    (29,1,4,2,35,28.0,0.8,6.5)
]

new_df = spark.createDataFrame(new_data, schema=feature_cols)

for c in feature_cols:
    new_df = new_df.withColumn(c, F.col(c).cast(DoubleType()))

new_assembled = assembler.transform(new_df)
new_scaled    = scaler_fit.transform(new_assembled)

new_preds = loaded_model.transform(new_scaled)

new_preds = new_preds.withColumn(
    "default_probability",
    vector_to_array("probability")[1]
)

new_pdf = new_preds.select(
    *feature_cols,
    "prediction",
    "default_probability"
).toPandas()

new_pdf['risk'] = new_pdf['default_probability'].apply(
    lambda p: "HIGH" if p>0.6 else (
        "MEDIUM" if p>0.35 else "LOW"
    )
)

print(new_pdf)

# Plot new predictions
plt.figure(figsize=(8,4))

colors = [
    "red" if p>0.6 else
    "orange" if p>0.35 else
    "steelblue"
    for p in new_pdf['default_probability']
]

bars = plt.bar(
    range(len(new_pdf)),
    new_pdf['default_probability'],
    color=colors
)

plt.axhline(
    loaded_model.getThreshold(),
    linestyle="--",
    color="black"
)

plt.title("New Sample Default Probability")
plt.ylabel("Probability")

plt.show()

# ── 4.12 Save Final Model
loaded_model.write().overwrite().save("logit_model_final")

print("SECTION 4 COMPLETE")
print("Final model saved : logit_model_final")